# Настройка окружения и загрузка данных

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import os

plt.style.use("seaborn-v0_8")
sns.set(font_scale=0.9)

os.makedirs("illustrations", exist_ok=True)

DATA_DIR = Path("../data")

ORIG_JSON_PATH = DATA_DIR / "original_data.json"
CLEAN_PATH = DATA_DIR / "cleaned_df.csv"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TRAIN_AUG_PATH = DATA_DIR / "train_augmented.csv"

TEXT_COL = "text"
LABEL_COL = "label"

# исходные данные (как есть из json)
with ORIG_JSON_PATH.open("r", encoding="utf-8") as f:
    original_data = json.load(f)
df_raw = pd.DataFrame(original_data)
df_clean = pd.read_csv(CLEAN_PATH)

# подготовленные обучающие и тестовые наборы
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)
df_train_augmented = pd.read_csv(TRAIN_AUG_PATH)

len(df_raw), len(df_clean), len(df_train), len(df_test), len(df_train_augmented)

(1774, 1755, 1404, 351, 2158)

In [18]:
# Fig 2.1 — распределение классов в исходном датасете (df_raw)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

counts = df_raw[LABEL_COL].value_counts().sort_values(ascending=False)
labels = [str(l) for l in counts.index]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(counts)), counts.values, color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Класс", fontsize=11)
ax.set_ylabel("Число документов", fontsize=11)
ax.set_title("Рисунок 2.1 — Распределение числа текстов по классам", fontsize=12, pad=10)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
ax.grid(axis="y", alpha=0.35, linestyle="--")
ax.grid(axis="y", which="minor", alpha=0.15, linestyle=":")
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1.5, str(int(h)),
            ha="center", va="bottom", fontsize=6.5, color="#333333")
plt.tight_layout()
plt.savefig("illustrations/fig2_1_class_distribution_raw.png", dpi=150)
plt.close()

In [19]:
# Fig 2.2 — распределение длины по символам (df_raw)
import matplotlib.pyplot as plt
import numpy as np

char_lengths = df_raw[TEXT_COL].astype(str).str.len()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(char_lengths, bins=60, color="#4C72B0", edgecolor="white", linewidth=0.3, alpha=0.85)
ax.axvline(char_lengths.median(), color="#C44E52", linewidth=1.5, linestyle="--",
           label=f"Медиана: {int(char_lengths.median())} симв.")
ax.axvline(char_lengths.mean(), color="#DD8452", linewidth=1.5, linestyle=":",
           label=f"Среднее: {int(char_lengths.mean())} симв.")
ax.set_xlabel("Число символов", fontsize=11)
ax.set_ylabel("Число документов", fontsize=11)
ax.set_title("Рисунок 2.2 — Распределение длины документов по числу символов", fontsize=12, pad=10)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.35, linestyle="--")
plt.tight_layout()
plt.savefig("illustrations/fig2_2_char_len_raw.png", dpi=150)
plt.close()

In [20]:
# Fig 2.3 — распределение по числу слов и приблизит. токенов (df_raw)
import matplotlib.pyplot as plt
import numpy as np

word_lengths = df_raw[TEXT_COL].astype(str).str.split().str.len()
# приближение токенов: ~1.4 токена на слово для русского
token_lengths = (word_lengths * 1.4).round().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, data, label, color in zip(
    axes,
    [word_lengths, token_lengths],
    ["Число слов", "Приблизительное число токенов"],
    ["#4C72B0", "#55A868"]
):
    ax.hist(data, bins=60, color=color, edgecolor="white", linewidth=0.3, alpha=0.85)
    ax.axvline(data.median(), color="#C44E52", linewidth=1.5, linestyle="--",
               label=f"Медиана: {int(data.median())}")
    ax.axvline(data.mean(), color="#DD8452", linewidth=1.5, linestyle=":",
               label=f"Среднее: {int(data.mean())}")
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel("Число документов", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.35, linestyle="--")
    # граница BERT 512 токенов
    if "токен" in label:
        ax.axvline(512, color="#8172B2", linewidth=1.2, linestyle="-.",
                   label="Лимит BERT (512)")
        ax.legend(fontsize=9)

axes[0].set_title("Число слов в документе", fontsize=11)
axes[1].set_title("Приблизительное число токенов", fontsize=11)
fig.suptitle("Рисунок 2.3 — Распределение длины документов (до очистки)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("illustrations/fig2_3_word_token_len_raw.png", dpi=150, bbox_inches="tight")
plt.close()

In [21]:
# Fig 2.4 — распределение по символам (df_clean)
import matplotlib.pyplot as plt
import numpy as np

char_lengths_clean = df_clean[TEXT_COL].astype(str).str.len()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(char_lengths_clean, bins=60, color="#55A868", edgecolor="white", linewidth=0.3, alpha=0.85)
ax.axvline(char_lengths_clean.median(), color="#C44E52", linewidth=1.5, linestyle="--",
           label=f"Медиана: {int(char_lengths_clean.median())} симв.")
ax.axvline(char_lengths_clean.mean(), color="#DD8452", linewidth=1.5, linestyle=":",
           label=f"Среднее: {int(char_lengths_clean.mean())} симв.")
ax.set_xlabel("Число символов", fontsize=11)
ax.set_ylabel("Число документов", fontsize=11)
ax.set_title("Рисунок 2.4 — Распределение длины документов по числу символов (после очистки)", fontsize=12, pad=10)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.35, linestyle="--")
plt.tight_layout()
plt.savefig("illustrations/fig2_4_char_len_clean.png", dpi=150)
plt.close()

In [22]:
# Fig 2.5 — по числу слов и токенов (df_clean)
import matplotlib.pyplot as plt
import numpy as np

word_lengths_clean = df_clean[TEXT_COL].astype(str).str.split().str.len()
token_lengths_clean = (word_lengths_clean * 1.4).round().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, data, label, color in zip(
    axes,
    [word_lengths_clean, token_lengths_clean],
    ["Число слов", "Приблизительное число токенов"],
    ["#55A868", "#4C72B0"]
):
    ax.hist(data, bins=60, color=color, edgecolor="white", linewidth=0.3, alpha=0.85)
    ax.axvline(data.median(), color="#C44E52", linewidth=1.5, linestyle="--",
               label=f"Медиана: {int(data.median())}")
    ax.axvline(data.mean(), color="#DD8452", linewidth=1.5, linestyle=":",
               label=f"Среднее: {int(data.mean())}")
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel("Число документов", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.35, linestyle="--")
    if "токен" in label:
        ax.axvline(512, color="#8172B2", linewidth=1.2, linestyle="-.",
                   label="Лимит BERT (512)")
        ax.legend(fontsize=9)

axes[0].set_title("Число слов в документе", fontsize=11)
axes[1].set_title("Приблизительное число токенов", fontsize=11)
fig.suptitle("Рисунок 2.5 — Распределение длины документов (после очистки)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("illustrations/fig2_5_word_token_len_clean.png", dpi=150, bbox_inches="tight")
plt.close()

In [23]:
# Fig 2.6 — распределение классов в train
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

counts_train = df_train[LABEL_COL].value_counts().sort_values(ascending=False)
labels_train = [str(l) for l in counts_train.index]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(counts_train)), counts_train.values,
              color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.set_xticks(range(len(labels_train)))
ax.set_xticklabels(labels_train, rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Класс", fontsize=11)
ax.set_ylabel("Число документов", fontsize=11)
ax.set_title("Рисунок 2.6 — Распределение классов в обучающей выборке", fontsize=12, pad=10)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
ax.grid(axis="y", alpha=0.35, linestyle="--")
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, str(int(h)),
            ha="center", va="bottom", fontsize=6.5, color="#333333")
plt.tight_layout()
plt.savefig("illustrations/fig2_6_class_dist_train.png", dpi=150)
plt.close()

In [24]:
# Fig 2.7 — распределение классов в test
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

counts_test = df_test[LABEL_COL].value_counts().sort_values(ascending=False)
labels_test = [str(l) for l in counts_test.index]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(counts_test)), counts_test.values,
              color="#C44E52", edgecolor="white", linewidth=0.4)
ax.set_xticks(range(len(labels_test)))
ax.set_xticklabels(labels_test, rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Класс", fontsize=11)
ax.set_ylabel("Число документов", fontsize=11)
ax.set_title("Рисунок 2.7 — Распределение классов в тестовой выборке", fontsize=12, pad=10)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
ax.grid(axis="y", alpha=0.35, linestyle="--")
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, str(int(h)),
            ha="center", va="bottom", fontsize=6.5, color="#333333")
plt.tight_layout()
plt.savefig("illustrations/fig2_7_class_dist_test.png", dpi=150)
plt.close()

In [25]:
# Fig 4.1 — распределение классов в train_augmented
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

counts_aug = df_train_augmented[LABEL_COL].value_counts().sort_values(ascending=False)
counts_orig = df_train[LABEL_COL].reindex(counts_aug.index).fillna(0)
labels_aug = [str(l) for l in counts_aug.index]
x = np.arange(len(labels_aug))
width = 0.42

fig, ax = plt.subplots(figsize=(16, 5))
bars_orig = ax.bar(x - width/2, counts_orig.values, width,
                   color="#4C72B0", alpha=0.75, edgecolor="white", linewidth=0.4, label="До аугментации")
bars_aug = ax.bar(x + width/2, counts_aug.values, width,
                  color="#55A868", alpha=0.85, edgecolor="white", linewidth=0.4, label="После аугментации")
ax.set_xticks(x)
ax.set_xticklabels(labels_aug, rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Класс", fontsize=11)
ax.set_ylabel("Число документов", fontsize=11)
ax.set_title("Рисунок 4.1 — Распределение классов в обучающей выборке после аугментации", fontsize=12, pad=10)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
ax.grid(axis="y", alpha=0.35, linestyle="--")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("illustrations/fig4_1_class_dist_augmented.png", dpi=150)
plt.close()